## PLUTO – Full Analysis

In this notebook, we perform a quantitative analysis on our final simulation runs.

### 0. Definitions

#### 0.1. Preamble

In [1]:
%%capture

%config InlineBackend.figure_formats = ['retina']

import os
import re
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=ResourceWarning)

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

import pyPLUTO as pp

from matplotlib.colors import LogNorm, SymLogNorm, TwoSlopeNorm

from scipy import signal
from scipy.ndimage import uniform_filter1d

import statsmodels.api as sm

plt.rcParams['axes.prop_cycle'] = plt.cycler(color=["steelblue", "olivedrab", "goldenrod", "firebrick", "rebeccapurple"])
plt.rcParams['figure.figsize'] = [8,5]
plt.rcParams['figure.constrained_layout.use'] = True
plt.rcParams['legend.frameon'] = False
plt.rcParams["xtick.minor.visible"] = True
plt.rcParams["ytick.minor.visible"] = True

plt.plot()
plt.close()

#### 0.2. Configuration

In [3]:
PATH = 'BH_VISC_RAD_HD_TRC_COMP_DF/'
LABEL = '120 x 90'
CACHE = os.path.join(PATH, 'Analysis_cache.npz')
FORCE_REDUCE = False

REF_PATH = 'BH_VISC_RAD_HD_TRC_COMP/'
REF_LABEL = '90 x 45'
REF_T_STEADY = None

T_MIN, T_MAX = None, None
DT_LONG = None
T_DENSE = None

T_FULL = [0.0, None]
T_EARLY = [50, 100, 150, 250, 400, 600]
T_EVOLUTION = [100, 1000, 10000, None]

W_WINDOW = 2500.0
BLOCK = 1000.0
REF_FRAC = 0.2
Z_CRIT = 2.0
T_STEADY_OVERRIDE = None
TOL_EQ = 0.2

R_WAVE = (4.0, 25.0)
WELCH_SEG = 6
WELCH_TSEG = 2500.0
R_PHASE_REF = 7.0
FIT_FRAC = 0.15
P_FUND_MIN = 0.3

R_PROFILES = [8.0, 12.0, 16.0]
PROFILE_STRIDE = 4

R_BUDGET = (2.5, 25.0)

HEMI = 2.0

#### 0.3. Setting

The system reproduces a toy model Active Galactic Nucleus (AGN) with a central supermassive black hole (SMBH) where its event horizon is realized as a diode modified outflow boundary, where a Paczyński–Wiita potential mimics the Schwarzschild radius. Surrounding the SMBH is a viscous and radiative accretion disk, initially seeded and continuously injected as the Kluźniak-Kita solution to the Shakura-Sunyaev prescription. Both viscosity and opacity are pinned to the disk and disabled in the corona via a custom classifier that reconstructs a Lagrangian tracer, which fails with radiation enabled in the Eulerian PLUTO code.

#### 0.4. Units

Our implementation uses $GM = c = 1$ conventions, where $M = M_\text{SMBH} = 10^8 M_\odot \simeq 2.0\times10^{41}\,\text{g}$ and

$$r_g = \frac{GM}{c^2} \simeq 1.5\times 10^{13} \simeq 1.0\,\text{AU} \qquad v_0 = \sqrt{\frac{GM}{r_g}} = c \simeq 3.0\times 10^{10}\,\text{cm}\,\text{s}^{-1} \qquad t_g = \frac{r_g}{c} = \frac{GM}{c^3} \simeq 490\,\text{s} \qquad \rho_0 \simeq 5.8\times 10^{-10}\,\text{g}\,\text{cm}^{-3}$$

are set as units, from which we derive

$$p_0 = \epsilon_0 = \rho_0 c^2 \simeq 5.2\times10^{11}\,\text{dyn}\,\text{cm}^{-2}\qquad \dot{M}_0 = \rho_0 c r_g^2 \simeq 3.8\times10^{27}\,\text{g}\,\text{s}^{-1} \simeq 60\,M_\odot\,\text{yr}^{-1}\qquad L_0 = \rho_0 c^3 r_g^2 \simeq 3.4\times 10^{48}\,\text{erg}\,\text{s}^{-1} \simeq 8.8\times 10^{14}\,L_\odot$$

for pressure or energy density, mass flux, and luminosity. This means that distances are given in gravitational radii $r_g$ and times in $t_g$ timescales throughout.

#### 0.5. Constants

The gas temperature follows from an ideal fully ionized gas

$$T_g = \frac{\mu m_u c^2}{k}\,\frac{p}{\rho} \qquad \mu = \bigl( 2X + \tfrac{3}{4} Y + \tfrac{1}{2} Z \bigr)^{-1}$$

where abundancies are set as

$$X = 0.70 \qquad Z = 0.02 \qquad Y = 1 - X - Z = 0.28$$

for the mean molecular weight. On the other hand, the radiation temperature follows from its energy density as

$$T_r = \left( \frac{E_r}{a} \right)^{\!1/4} \qquad a = \frac{\sigma_\text{SB}}{\rho_0 c^3}$$

with the Stefan-Boltzmann constant. Radiation pressure is $p_r = E_r / 3$ in the optically thick limit, and PLUTO stores its flux $F$ in units of energy density, meaning $cF$ gives the physical flux.

#### 0.6. Model

The computational grid uses spherical $(r, \vartheta)$ coordinates, where $R = r\sin\vartheta$ and $z = r\cos\vartheta$ are cylindrical quantities, as well as axisymmetry and equatorialsymmetry, so that $\vartheta = [0, \pi / 2]$ gives the angular domain. Mechanics are Newtonian with a Paczyński–Wiita potential

$$\Phi(r) = -\frac{GM}{r - r_\text{S}} = -\frac{1}{r - 2}$$

with $r_\text{S} = 2r_g$ the Schwarzschild radius. Circular orbits in the midplane have

$$\Omega^2 = \frac{1}{R}\frac{\mathrm{d}\Phi}{\mathrm{d}R} = \frac{1}{R\,(R-2)^2}\qquad \ell = R^2\,\Omega\qquad \kappa^2 = \frac{1}{R^3}\frac{\mathrm{d}\left(R^4\Omega^2\right)}{\mathrm{d}R} = \Omega^2\,\frac{R - 6}{R - 2}$$

as angular velocity, specific angular momentum and epicyclic frequency. From this follows an innermost stable orbit (ISCO) defined by $\kappa^2 = 0$ at $R_\text{isco} = 6r_g$ from the singularity. The Newtonian Keplerian angular velocity $\Omega = R^{-3/2}$ is still used for the initial disk seeding as well as the disk versus corona classifier.

An adiabatic atmosphere in hydrostatic equilibrium with the Newtonian potential $\Phi(r) = -1/r$ and for $\gamma = C_p / C_v = 5/3$ adiabatic index

$$\rho_c(r) = \rho_c^{(0)}\, r^{-3/2} \qquad p_c(r) = \frac{2}{5}\,\rho_c^{(0)}\, r^{-5/2} \qquad \frac{p_c}{\rho_c} = \frac{\gamma-1}{\gamma}\,\frac{1}{r}$$

defines the corona and ensures numerical stability. The Kluźniak & Kita polytropic disk with aspect ratio $\epsilon = H/R$ follows from

$$C(r,\vartheta) = \frac{2}{5\epsilon^2}\left[\frac{1}{r} - \left(1 - \frac{5}{2}\epsilon^2\right)\frac{1}{R}\right] \qquad \rho_d = C^{3/2} \qquad p_d = \epsilon^2\, C^{5/2}$$

and occupies all cells where $p_d \geq p_c$ and $R > R_d$ hold. Its velocities include the viscous inflow of the $\alpha$ disk

$$v_\varphi = \frac{1}{\sqrt{R}}\left[\sqrt{1 - \frac{5}{2}\epsilon^2} + \frac{2}{3}\epsilon^2\alpha^2\lambda\left(1 - \frac{6}{5\,\epsilon^2\tan^2\vartheta}\right)\right]\qquad v_r = -\frac{\alpha\,\epsilon^2}{\sin\vartheta\,\sqrt{R}}\left[10 - \frac{32}{3}\lambda\alpha^2 - \lambda\left(5 - \frac{1}{\epsilon^2\tan^2\vartheta}\right)\right]$$

with $\lambda = \dfrac{11}{5}\left(1 + \dfrac{64}{25}\alpha^2\right)^{-1}$ and $v_\vartheta = 0$ while in the corona all velocities vanish.

Radiation is initially seeded to share the analytic disk pressure with the gas at a common temperature

$$\frac{\rho\,T}{K_T} + \frac{a\,T^4}{3} = p_d \qquad p_\mathrm{g} = \frac{\rho\,T}{K_T} \qquad E_r = a\,T^4 \qquad \boldsymbol{F} = 0$$

where $K_T = \mu m_u c^2/k$ is used, and the corona gets assigned an initial $E_r = 10^{-16}$ corresponding to the minimun radiation energy.

Only the shear viscosity $\nu_1$ of an $\alpha$ prescription with the initial analytic sound speed acts

$$\nu_1 = \frac{2}{3}\,\rho\,\alpha\, c_s^2\, R^{3/2}\, f \qquad c_s^2 = \epsilon^2 \max(C, 0)$$

while the bulk viscosity $\nu_2$ is neglected

$$\nu_2 = 0$$

where $f$ is the tracer replacement classifier. In the midplane $C = 1/R$ so the kinematic viscosity is

$$\nu(R) = \frac{\nu_1}{\rho} = \frac{2}{3}\,\alpha\,\epsilon^2\, R^{1/2}$$

and the viscous heating rate per volume is $q^+ = \nu_1 \left(R\,\partial\Omega/\partial R\right)^2$ from the dominant $r\varphi$ stress. Since the analytic sound speed is fixed, viscous heating does not respond to the thermal state of the disk, which suppresses the thermal and viscous instability of radiation pressure dominated disks. A self consistent $\alpha$ prescription uses the local total pressure for

$$\nu_1 = \frac{2}{3}\,\rho\,\alpha\,c_s^2\,R^{3/2} f \qquad c_s^2 = \frac{p + E_r/3}{\rho}$$

capped in hot and thin gas.

A continuous classifier $f \in [0,1]$ multiplies a density and a rotation criterion

$$f = \sigma\!\left(\frac{s - 1/6}{0.3}\right)\,\sigma\!\left(\frac{|v_\varphi|\sqrt{R} - 0.5}{0.1}\right) \qquad
s = \frac{\log\rho - \log\bar\rho_c(r)}{\max\left[\log(\bar\rho_d/\bar\rho_c),\ 0.5\right]} \qquad \sigma(x) = \frac{1}{1 + e^{-x}}$$

with reference profiles $\bar\rho(r)$ of the corona and the disk in logarithmic radial bins. They are volume averages weighted by $f$ of each bin or, optionally, the bin maximum density and the geometric mean density above $z/R = 0.5$ to remove the self referntial feature of the method. New estimates $\hat\rho$ enter through an exponential moving average with time constant $\tau = 6\,t_g$ via

$$\bar\rho^{\,n+1} = \alpha\,\hat\rho^{\,n+1} + (1 - \alpha)\,\bar\rho^{\,n} \qquad \alpha = 1 - e^{-\Delta t_u/\tau}$$

with $\Delta t_u$ the time between updates. An estimate therefore decays as $e^{-(t - t')/\tau}$ independent of resolution and update
cadence. Bins whose span falls below $0.5\,\text{dex}$ take the references of the nearest bin with a sufficient span. Previous runs used a fixed $\alpha = 0.05$ per step without these guards.

With $\rho$ and $T$ in CGS as well as a $T \geq 10^4\,\text{K}$ clamped gas temperature, we get

$$\kappa_\mathrm{es} = 0.20\,(1+X)\,\mathrm{cm^2\,g^{-1}}$$

for electron scattering and

$$\kappa_\mathrm{ff+bf} = \left(K_\mathrm{ff} + K_\mathrm{bf}\right) \rho\, T^{-7/2} \qquad K_\mathrm{ff} = 3.68 \times 10^{22}\,(1-Z)(1+X) \qquad K_\mathrm{bf} = 4.34 \times 10^{25}\, \frac{Z(1+X)}{10}$$

following Kramers law. Compton energy exchange enters as an effective absorption opacity

$$\kappa_C = \kappa_\mathrm{es}\, \frac{4 k T_r^4}{m_e c^2\,(T_g + T_r)(T_g^2 + T_r^2)} \;\; (T_g \leq T_\mathrm{max}) \qquad
\kappa_C = \kappa_\mathrm{es}\, \frac{4 k T_r^4\,(T_\mathrm{max} - T_r)}{m_e c^2\,(T_g^4 - T_r^4)} \;\; (T_g > T_\mathrm{max})$$

that with $T_\mathrm{max} = 10^9\,\text{K}$ reproduces the nonrelativistic rate $\rho\,\kappa_\mathrm{es} c\, E_r\, 4k (T_g - T_r)/(m_e c^2)$ exactly. We define

$$\kappa_a = g_v\, f_d \left(\kappa_\mathrm{ff+bf} + \kappa_C\right) \qquad \kappa_s = g_v \left(\kappa_\mathrm{es} - f_d\, \kappa_C\right) \qquad f_d = \max\!\left(0, \frac{f - 0.05}{0.95}\right) \qquad g_v = \sigma\!\left(-\frac{|\boldsymbol{v}|/c - 0.9}{0.03}\right)$$

such that absorption and Compton act only in the disk, while Thomson scattering applies everywhere, and $g_v$ decouples matter and radiation in the plunging region, where the gas moves faster than the nonrelativistic radiation module can handle. Opacities are per unit mass and converted with $\rho_0 R_g$ to code units.

The Eddington luminosity and accretion rate are given as

$$L_\mathrm{Edd} = \frac{4\pi G M c}{\kappa_\mathrm{es}} \qquad\dot M_\mathrm{Edd} = \frac{L_\mathrm{Edd}}{\eta\, c^2}$$

where $\eta = 1/16$ derives from the binding energy at the ISCO of the Paczyński–Wiita potential.

In this PLUTO build radiation is not advected with the gas, does no work on it and exerts no drag. Only the comoving source terms
$\rho\kappa_a c\,(aT_g^4 - E_r)$ and $\rho(\kappa_a+\kappa_s)\boldsymbol{F}$ couple the two. Compression of the gas does not heat the
radiation, which matters for the wave diagnostics.

At the inner domain boundary, all variables are copied from the first active cell, a diode enforces $v_r \leq 0$ and also floors $\rho \geq \rho_\mathrm{fl}$ and $p \geq 10^{-3}\rho_\mathrm{fl}$ while radiation $E_r$ is copied and $F_r \leq 0$ is set. Beyond the outer edge, density is extrapolated as $\rho_i = \rho_{i-1}\,(r_i/r_{i-1})^{a}$ with a Van Leer limited, negative slope $a$ and a pressure adiabatic $p = p_N\,(\rho/\rho_N)^\gamma$ while $v_r$ and $v_\vartheta$ are copied, and $v_\varphi$ is extrapolated. Within $|\vartheta - \pi/2| \leq \arctan(1.25\,\epsilon)$ the analytic disk density and velocities are reinjected. Within $|\vartheta - \pi/2| \geq \arctan(3\,\epsilon)$ coronal inflow is blocked by setting velocities to zero where $v_r < 0$ is fulfilled. Radiation $E_r$ gets copied and $F_r \geq 0$ is used again. The grid at $\vartheta = 0$ is axisymmetric and takes on at $\vartheta = \pi/2$ equatorial symmetry. Inside the domain, density is $\rho \geq \rho_\mathrm{fl} = 5\times10^{-7}$ which thereby raises $\rho$ at fixed sound speed and momentum, and $T_g \geq 10^4\ \mathrm{K}$ is set. Both floors add mass or energy, which can be quantified.

#### 0.7. Characteristics

For a thin disk with $H = \epsilon R$ and midplane sound speed $c_s^{(0)} = \epsilon\,R^{-1/2}$ the relevant timescales are

$$
t_\mathrm{orb} = \frac{2\pi}{\Omega} \qquad
t_\mathrm{th} = \frac{1}{\alpha\,\Omega_N} \qquad
t_\nu = \frac{R^2}{\nu} = \frac{3}{2}\,\frac{R^{3/2}}{\alpha\,\epsilon^2} \qquad
t_\mathrm{sound} = \frac{R}{c_s^{(0)}} \qquad
t_\mathrm{diff} \simeq \frac{\tau\, H}{c}
$$

where $$\tau = \frac{\kappa\,\Sigma}{2}$$ 

defines the optical depth from the midplane to the disk surface. The viscous time sets how far out the disk can reach inflow equilibrium within a given run time as $R_\nu(t)$ with $t_\nu(R_\nu) = t$ while the diffusion time needs the measured surface density.